# Batch Folder Inference: Plate -> Text/Province -> CSV

โน้ตบุ๊กนี้ใช้ pipeline เดียวกับ `pipe_load.ipynb` เพื่อ:
- เลือกโฟลเดอร์รูปภาพ
- รัน Plate Detector -> Plate Splitter -> OCR + Province Classifier
- บันทึกผลเป็น CSV โดยมีชื่อไฟล์และผลที่อ่านได้

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime
import time
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

import torch
import torch.nn.functional as F
import timm
import torchvision.transforms as T
from PIL import Image

# ------------------------------
# Paths to weights (ปรับได้ถ้าไฟล์ย้าย)
WEIGHTS_DIR = Path(r"D:\CodingD\ALPR\weights")
PLATE_DET_W = WEIGHTS_DIR / "plate_detector_best.pt"
PLATE_SPLIT_W = WEIGHTS_DIR / "plate_splitter_best.pt"
PROVINCE_CKPT = WEIGHTS_DIR / "province_classifier_best_new_model.pt"
OCR_CKPT = WEIGHTS_DIR / "upper_ctc_special_best.pt"

# ------------------------------
# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ------------------------------
# Province classifier helpers
province_tfm = T.Compose([
    T.Resize((32, 128)),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

def load_province_classifier(ckpt_path: Path):
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Province checkpoint not found: {ckpt_path}")
    ckpt = torch.load(str(ckpt_path), map_location=device)
    model = timm.create_model(
        ckpt["model_name"],
        pretrained=False,
        num_classes=ckpt["num_classes"],
        in_chans=3,
    ).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()

    idx2label = ckpt.get("idx2label", None)
    if idx2label is None:
        label2idx = ckpt.get("label2idx", {})
        idx2label = {int(v): k for k, v in label2idx.items()}
    else:
        idx2label = {int(k): v for k, v in idx2label.items()}

    return model, idx2label

@torch.no_grad()
def predict_province_from_bgr(bgr_crop: np.ndarray, province_model, idx2label, topk: int = 3):
    if bgr_crop is None or bgr_crop.size == 0:
        return []
    rgb = cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb)
    x = province_tfm(img).unsqueeze(0).to(device)
    logits = province_model(x)
    probs = F.softmax(logits, dim=1).squeeze(0)

    k = min(topk, probs.numel())
    vals, idxs = torch.topk(probs, k=k)
    out = []
    for v, i in zip(vals.cpu().tolist(), idxs.cpu().tolist()):
        out.append((idx2label.get(int(i), str(int(i))), float(v)))
    return out

# ------------------------------
# OCR helpers (CRNN + CTC)
class CRNN(torch.nn.Module):
    def __init__(self, num_classes: int, img_height: int = 32):
        super().__init__()
        self.cnn = torch.nn.Sequential(
            torch.nn.Conv2d(1, 64, 3, 1, 1), torch.nn.BatchNorm2d(64), torch.nn.ReLU(True),
            torch.nn.MaxPool2d(2, 2),
            torch.nn.Conv2d(64, 128, 3, 1, 1), torch.nn.BatchNorm2d(128), torch.nn.ReLU(True),
            torch.nn.MaxPool2d(2, 2),
            torch.nn.Conv2d(128, 256, 3, 1, 1), torch.nn.BatchNorm2d(256), torch.nn.ReLU(True),
            torch.nn.Conv2d(256, 256, 3, 1, 1), torch.nn.BatchNorm2d(256), torch.nn.ReLU(True),
            torch.nn.MaxPool2d((2, 1), (2, 1)),
            torch.nn.Conv2d(256, 512, 3, 1, 1), torch.nn.BatchNorm2d(512), torch.nn.ReLU(True),
            torch.nn.MaxPool2d((2, 1), (2, 1)),
        )
        self.rnn = torch.nn.LSTM(
            512 * (img_height // 16),
            256,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = torch.nn.Linear(256 * 2, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.cnn(x)
        b, c, h, w = feats.size()
        feats = feats.permute(0, 3, 1, 2).contiguous()
        feats = feats.view(b, w, c * h)
        rnn_out, _ = self.rnn(feats)
        logits = self.classifier(rnn_out)
        return logits.permute(1, 0, 2)

def load_ocr_model(ckpt_path: Path):
    if not ckpt_path.exists():
        raise FileNotFoundError(f"OCR checkpoint not found: {ckpt_path}")
    ckpt = torch.load(str(ckpt_path), map_location=device)
    idx_to_char = ckpt.get("idx_to_char", None)
    if idx_to_char is None:
        raise ValueError("idx_to_char not found in OCR checkpoint")

    model = CRNN(num_classes=len(idx_to_char)).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model, idx_to_char

ocr_tfm = T.Compose([
    T.Resize((32, 128)),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,)),
])

@torch.no_grad()
def ocr_greedy_decode(logits: torch.Tensor, ocr_idx_to_char) -> str:
    probs = logits.softmax(2)
    indices = probs.argmax(2).permute(1, 0)
    seq = indices[0].tolist()

    prev = None
    chars = []
    for idx in seq:
        if idx != 0 and idx != prev:
            chars.append(ocr_idx_to_char[idx])
        prev = idx
    return "".join(chars)

@torch.no_grad()
def predict_text_from_bgr(bgr_crop: np.ndarray, ocr_model, ocr_idx_to_char):
    if bgr_crop is None or bgr_crop.size == 0:
        return ""
    gray = cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2GRAY)
    img = Image.fromarray(gray)
    x = ocr_tfm(img).unsqueeze(0).to(device)
    logits = ocr_model(x)
    return ocr_greedy_decode(logits, ocr_idx_to_char)

# ------------------------------
# Generic image loader (รองรับ path ไทย/ยูนิโค้ด)
def load_bgr(path: Path):
    img = cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Cannot load image: {path}")
    return img

# ------------------------------
# Load all models once
plate_model = YOLO(PLATE_DET_W)
splitter_model = YOLO(PLATE_SPLIT_W)
province_model, idx2label = load_province_classifier(PROVINCE_CKPT)
ocr_model, ocr_idx_to_char = load_ocr_model(OCR_CKPT)

print("Models loaded successfully.")

Device: cuda
Models loaded successfully.


In [2]:
from tkinter import Tk, filedialog

# ------------------------------
# 1) เลือกโฟลเดอร์รูปภาพ
root = Tk()
root.withdraw()
root.attributes("-topmost", True)
selected_dir = filedialog.askdirectory(title="เลือกโฟลเดอร์รูปภาพสำหรับรัน ALPR")
root.destroy()

if not selected_dir:
    raise ValueError("ยังไม่ได้เลือกโฟลเดอร์รูปภาพ")

input_dir = Path(selected_dir)
print("Selected folder:", input_dir)

# ------------------------------
# 2) รวบรวมไฟล์ภาพ
exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
image_paths = [p for p in input_dir.rglob("*") if p.suffix.lower() in exts]
image_paths = sorted(image_paths)

if not image_paths:
    raise FileNotFoundError(f"ไม่พบไฟล์ภาพในโฟลเดอร์: {input_dir}")

print(f"Found {len(image_paths)} images")

# ------------------------------
# 3) Batch inference
rows = []
start_all = time.perf_counter()

for i, img_path in enumerate(image_paths, start=1):
    try:
        frame = load_bgr(img_path)
        t0 = time.perf_counter()

        # Step 1: plate detector
        det = plate_model.predict(frame, conf=0.25, iou=0.7, imgsz=1280, verbose=False)[0]
        plate_boxes = det.boxes.xyxy.cpu().numpy() if det.boxes is not None else []
        plate_confs = det.boxes.conf.cpu().numpy() if det.boxes is not None else []

        if len(plate_boxes) == 0:
            rows.append({
                "file_name": img_path.name,
                "file_path": str(img_path),
                "plate_found": 0,
                "plate_conf": 0.0,
                "plate_text": "",
                "province_pred": "",
                "province_conf": 0.0,
                "status": "no_plate",
                "latency_ms": round((time.perf_counter() - t0) * 1000, 2),
            })
            continue

        best_idx = int(np.argmax(plate_confs))
        x1, y1, x2, y2 = map(int, plate_boxes[best_idx])
        plate_crop = frame[y1:y2, x1:x2]

        if plate_crop.size == 0:
            rows.append({
                "file_name": img_path.name,
                "file_path": str(img_path),
                "plate_found": 0,
                "plate_conf": float(plate_confs[best_idx]),
                "plate_text": "",
                "province_pred": "",
                "province_conf": 0.0,
                "status": "invalid_plate_crop",
                "latency_ms": round((time.perf_counter() - t0) * 1000, 2),
            })
            continue

        # Step 2: splitter
        sp = splitter_model.predict(plate_crop, conf=0.25, iou=0.6, imgsz=640, verbose=False)[0]
        split_boxes = sp.boxes.xyxy.cpu().numpy() if sp.boxes is not None else []
        split_cls = sp.boxes.cls.cpu().numpy().astype(int) if sp.boxes is not None else []
        split_confs = sp.boxes.conf.cpu().numpy() if sp.boxes is not None else []

        text_pred = ""
        province_pred = ""
        province_conf = 0.0

        # class 0 = license_text
        text_idxs = [j for j, c in enumerate(split_cls) if int(c) == 0]
        if text_idxs:
            best_t = max(text_idxs, key=lambda j: split_confs[j])
            x1t, y1t, x2t, y2t = map(int, split_boxes[best_t])
            text_crop = plate_crop[y1t:y2t, x1t:x2t]
            text_pred = predict_text_from_bgr(text_crop, ocr_model, ocr_idx_to_char)

        # class 1 = province
        prov_idxs = [j for j, c in enumerate(split_cls) if int(c) == 1]
        if prov_idxs:
            best_p = max(prov_idxs, key=lambda j: split_confs[j])
            x1p, y1p, x2p, y2p = map(int, split_boxes[best_p])
            prov_crop = plate_crop[y1p:y2p, x1p:x2p]
            top = predict_province_from_bgr(prov_crop, province_model, idx2label, topk=1)
            if top:
                province_pred, province_conf = top[0][0], float(top[0][1])

        rows.append({
            "file_name": img_path.name,
            "file_path": str(img_path),
            "plate_found": 1,
            "plate_conf": float(plate_confs[best_idx]),
            "plate_text": text_pred,
            "province_pred": province_pred,
            "province_conf": province_conf,
            "status": "ok",
            "latency_ms": round((time.perf_counter() - t0) * 1000, 2),
        })

        if i % 50 == 0:
            print(f"Processed {i}/{len(image_paths)}")

    except Exception as e:
        rows.append({
            "file_name": img_path.name,
            "file_path": str(img_path),
            "plate_found": 0,
            "plate_conf": 0.0,
            "plate_text": "",
            "province_pred": "",
            "province_conf": 0.0,
            "status": f"error: {e}",
            "latency_ms": 0.0,
        })

elapsed = time.perf_counter() - start_all

# ------------------------------
# 4) Save CSV
result_df = pd.DataFrame(rows)
out_dir = Path("pipeline") / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / f"pipeline_detector_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
result_df.to_csv(out_csv, index=False, encoding="utf-8-sig")

print("Done.")
print("Output CSV:", out_csv)
print(result_df.head(10))
print(f"Total images: {len(result_df)} | Total time: {elapsed:.2f}s | Avg: {(elapsed/max(1, len(result_df)))*1000:.1f} ms/image")

Selected folder: C:\Users\PC\Downloads\for ALPR sample\for ALPR sample
Found 30 images
Done.
Output CSV: pipeline\outputs\pipeline_detector_results_20260330_173236.csv
        file_name                                          file_path  \
0  08_00_00.6.jpg  C:\Users\PC\Downloads\for ALPR sample\for ALPR...   
1  08_00_32.9.jpg  C:\Users\PC\Downloads\for ALPR sample\for ALPR...   
2  08_01_06.7.jpg  C:\Users\PC\Downloads\for ALPR sample\for ALPR...   
3  08_04_52.9.jpg  C:\Users\PC\Downloads\for ALPR sample\for ALPR...   
4  08_04_54.1.jpg  C:\Users\PC\Downloads\for ALPR sample\for ALPR...   
5  08_04_55.9.jpg  C:\Users\PC\Downloads\for ALPR sample\for ALPR...   
6  08_05_21.8.jpg  C:\Users\PC\Downloads\for ALPR sample\for ALPR...   
7  08_05_29.1.jpg  C:\Users\PC\Downloads\for ALPR sample\for ALPR...   
8  08_05_47.1.jpg  C:\Users\PC\Downloads\for ALPR sample\for ALPR...   
9  08_09_14.4.jpg  C:\Users\PC\Downloads\for ALPR sample\for ALPR...   

   plate_found  plate_conf plate_text  